# PLSSVD evaluation


Computational scripts save assignment `-1` first, then random assignments `0–99`.
`PAIRING_VIEW` changes only the loaded/displayed results. Full results exclude `-1`
and give each random assignment equal weight after averaging its internal folds.
Mean ± SD are descriptive, not confidence intervals. Missing assignments are an
error; assignment zero and incomplete batches are never substituted for the full result.
Raw signal preparation runs once in a terminal and includes every eligible MEG subject.

Temporal surrogate permutations and repeated trial folds are independent inner axes. At least two inner trial iterations are needed by the existing component-selection procedure.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
PAIRING_VIEW = 'matched'  # 'matched': -1; 'full': mean ± SD of IDs 0–99.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, pairing_context
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED, display_only=True)
import vs0_load_and_config as shared_data
from vs0_pairing_results import config_view, path_view, show, safe_load, ResultView
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
DISPLAY_COMPONENTS = 5
SEED = 2026  # Trial/source-control seed, independent of PAIRING_SEED.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MAX_GRAM_GIB = 2.
from vs1_coverage_figures import apply_style, MODALITY_COLORS, SETUP_COLORS
apply_style()

from vs0_pairing_results import evaluation_view,pca_comparison_view,permutation_view,selection_view
MEG_KIND='paired_coverage'
N_COMPONENTS=100
MAX_COMPONENTS=100
N_FIT_ITERATIONS=2  # Inner repeated trial-fold evaluations, not subject assignments.
N_ITERATIONS=None  # Number of inner evaluation iterations to display; None means all.
N_SPLITS=5
TEMPORAL_PERMUTATION_TYPE='time_point'
PERMUTATION_MODES=['meg','ieeg','both']
SELECTION_PERM='both'
PCA_ABSOLUTE=True
SCALING_MODE='equal_variance' if BLOCK_SCALING else 'none'
CACHE_DIR=shared_data.trial_cache_directory(ROOT)
RUNS_DIR=ANALYSIS_OUTPUT_ROOT/('plssvd_eval_equal_var' if BLOCK_SCALING else 'plssvd_eval')/MEG_KIND
RECOMPUTE=False
from vs2_1_plssvd_eval_utils import validation_run_name

## Fit, prepare and load unpermuted evaluation

In [ ]:
INPUT_DIR=RUNS_DIR/'none'
fit_command=['python','-u',str(ROOT/'vs2_1_plssvd_eval.py'),'--root',str(ROOT),
    '--output-dir',str(INPUT_DIR),'--cache-dir',str(CACHE_DIR),'--meg-kind',MEG_KIND,
    '--block-scaling',SCALING_MODE,'--max-components',str(MAX_COMPONENTS),
    '--n-iterations',str(N_FIT_ITERATIONS),'--n-splits',str(N_SPLITS),'--seed',str(SEED),
    '--perm','none','--perm-type','none']
post_command=['python','-u',str(ROOT/'vs2_1_plssvd_postprocess.py'),'--runs-dir',str(RUNS_DIR),
    '--components',str(N_COMPONENTS),'--perm','none','--perm-type','none']
summary_command=['python','-u',str(ROOT/'vs2_1_plssvd_results.py'),'--run-dir',str(INPUT_DIR),'--components',str(N_COMPONENTS)]
if N_ITERATIONS is not None:summary_command+=['--iterations',str(N_ITERATIONS)]
for command in (fit_command,post_command,summary_command):print(format_command(command))
evaluation=safe_load(evaluation_view,INPUT_DIR,N_COMPONENTS,PAIRING_VIEW,N_PAIRINGS,N_ITERATIONS,
                     meg_kind=MEG_KIND,block_scaling=SCALING_MODE)

## Reconstruction, prediction, covariance, correlation, components and spectral metrics

Every prepared scalar metric is included. Full summaries average folds/inner evaluations within each subject assignment first.

In [ ]:
if evaluation is not None:
    show(evaluation, max_components=DISPLAY_COMPONENTS, tables=True, arrays=False)

## Time courses, covariance and correlation matrices, spectra, and PCA diagnostics

Full panels report the mean and SD of assignment-level arrays. Component-rank diagnostics retain native rank; score traces and signed covariance/correlation displays use correspondence to the matched assignment.

In [ ]:
if evaluation is not None:
    show(evaluation, max_components=DISPLAY_COMPONENTS, tables=False, arrays=True)

## Comparison with full-data PCA

In [ ]:
command=['python','-u',str(ROOT/'vs2_1_plssvd_fit_pca.py'),'--run-dir',str(INPUT_DIR),'--cache-dir',str(CACHE_DIR),
    '--meg-kind',MEG_KIND,'--block-scaling',SCALING_MODE,'--components',str(N_COMPONENTS),
    '--absolute' if PCA_ABSOLUTE else '--no-absolute']
if N_ITERATIONS is not None:command+=['--iterations',str(N_ITERATIONS)]
print(format_command(command))
full_pca=safe_load(pca_comparison_view,INPUT_DIR,N_COMPONENTS,PAIRING_VIEW,N_PAIRINGS,
                  meg_kind=MEG_KIND,block_scaling=SCALING_MODE,absolute=PCA_ABSOLUTE,n_iterations=N_ITERATIONS)

In [ ]:
if full_pca is not None:
    show(full_pca, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)

## Temporal permutation effects

The same outer subject IDs and pairing seed are used for baseline and null runs. time_point remains a temporal surrogate, never a subject-assignment operation. Each null is fitted and postprocessed for -1 and 0–99.

In [ ]:
null_comparisons={}
for mode in PERMUTATION_MODES:
    name=validation_run_name(mode,TEMPORAL_PERMUTATION_TYPE)
    command=fit_command.copy()
    command[command.index('--output-dir')+1]=str(RUNS_DIR/name)
    command[command.index('--perm')+1]=mode
    command[command.index('--perm-type')+1]=TEMPORAL_PERMUTATION_TYPE
    print(format_command(command))
    post=['python','-u',str(ROOT/'vs2_1_plssvd_postprocess.py'),'--runs-dir',str(RUNS_DIR),
          '--components',str(N_COMPONENTS),'--perm',mode,'--perm-type',TEMPORAL_PERMUTATION_TYPE]
    print(format_command(post))
    null_comparisons[mode]=safe_load(permutation_view,RUNS_DIR,name,N_COMPONENTS,PAIRING_VIEW,N_PAIRINGS)

In [ ]:
for mode,result in null_comparisons.items():
    if result is not None:
        print('Temporal null:',mode,TEMPORAL_PERMUTATION_TYPE)
        show(result,max_components=DISPLAY_COMPONENTS)

## Component selection

The existing null-quantile/support rule is applied separately within each subject assignment. Full mode summarizes every resulting metric, including selected dimension and per-rank retention frequency; it does not turn their mean into a new integer selection rule.

In [ ]:
command=['python','-u',str(ROOT/'vs2_1_plssvd_select_components.py'),'--runs-dir',str(RUNS_DIR),
    '--components',str(N_COMPONENTS),'--perm',SELECTION_PERM,'--perm-type',TEMPORAL_PERMUTATION_TYPE]
if RECOMPUTE:command.append('--recompute')
print(format_command(command))
selection=safe_load(selection_view,RUNS_DIR,N_COMPONENTS,SELECTION_PERM,TEMPORAL_PERMUTATION_TYPE,PAIRING_VIEW,N_PAIRINGS)

In [ ]:
if selection is not None:
    show(selection, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)